# Figure 5 | Cedar Rapids ERT processing and inversion

Run the cells in order from the repository root or `examples/`. Panels are saved to `examples/Figure/fig5/` at 400 dpi. The source field data and saved case results must be obtained separately. The assembled manuscript figure is not generated here.

The upper workflow ribbon is assembled manually. Panels include the poor-fit Ellis CW1 result; its retrospective Agent verdict is a veto, so its section is a limited comparison.

## Setup and saved AEM inputs

In [ ]:
from pathlib import Path
import json
import os
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, ListedColormap, BoundaryNorm
from matplotlib.cm import ScalarMappable
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MaxNLocator, FuncFormatter
from scipy.spatial import cKDTree
from scipy.interpolate import griddata
from PIL import Image

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'examples').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or examples directory.')


ROOT = find_project_root()
CASE = ROOT / 'Result' / 'Cedar-Rapids'
OUT = ROOT / 'examples' / 'Figure' / 'fig5'
OUT.mkdir(parents=True, exist_ok=True)

STATE = json.loads((CASE / 'case_state.json').read_text(encoding='utf-8'))
AEM_RUN = CASE / 'runs' / STATE['aem_run_id']
GEO_MODEL = CASE / 'imports' / 'geological_model.npz'

mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 11,
    'axes.labelsize': 14, 'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'axes.linewidth': 1.1, 'axes.grid': False,
    'legend.frameon': False, 'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
})

RES_CMAP = mpl.colormaps['turbo'].copy()
RES_CMAP.set_bad('white')
RES_NORM = LogNorm(1.0, 1.0e4)
RES_TICKS = [1, 10, 100, 1000, 10000]
RES_TICKLABELS = [r'$10^0$', r'$10^1$', r'$10^2$', r'$10^3$', r'$10^4$']

UNIT_COLORS = np.array([
    mpl.colors.to_rgba('#CDD1D5', 1.00),  # below DOI: visually complete, uninterpreted
    mpl.colors.to_rgba('#E8D8AD', 1.00),  # above rise
    mpl.colors.to_rgba('#D95F47', 1.00),  # resistive domain
    mpl.colors.to_rgba('#4C78A8', 1.00),  # finer fill
])
UNIT_CMAP = ListedColormap(UNIT_COLORS)
UNIT_NORM = BoundaryNorm(np.arange(-0.5, 4.5, 1.0), UNIT_CMAP.N)

def bold_axes(ax):
    ax.xaxis.label.set_fontweight('bold')
    ax.yaxis.label.set_fontweight('bold')
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.tick_params(width=1.2, length=5)

def save_png(fig, filename, *, tight=True):
    path = OUT / filename
    kwargs = {'bbox_inches': 'tight'} if tight else {}
    fig.savefig(path, dpi=400, facecolor='white', **kwargs)
    # Optional vector copy for rendered-text QA; off unless the variable is set.
    qa_dir = os.environ.get('FIG5_QA_PDF_DIR')
    if qa_dir:
        fig.savefig(Path(qa_dir) / Path(filename).with_suffix('.pdf'), facecolor='white', **kwargs)
    plt.close(fig)
    return path

def add_resistivity_colorbar(fig, cax, mappable):
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks(RES_TICKS)
    cb.set_ticklabels(RES_TICKLABELS)
    cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
    cb.ax.tick_params(labelsize=11, width=1.2, length=5)
    for label in cb.ax.get_yticklabels():
        label.set_fontweight('bold')
    return cb

def resistivity_rgba(values):
    values = np.asarray(values, float)
    clipped = np.clip(np.nan_to_num(values, nan=1.0), 1.0, 1.0e4)
    return RES_CMAP(RES_NORM(clipped.ravel())).reshape(values.shape + (4,))

def cell_edges(centres):
    centres = np.asarray(centres, dtype=float)
    delta = np.diff(centres)
    return np.r_[centres[0] - delta[0] / 2,
                 (centres[:-1] + centres[1:]) / 2,
                 centres[-1] + delta[-1] / 2]

def style_3d(ax):
    ax.set_proj_type('ortho')
    ax.view_init(elev=25, azim=-58)
    ax.set_box_aspect((2.0, 1.0, 0.72))
    ax.grid(False)
    ax.set_xlabel('Easting (km)', labelpad=20, fontweight='bold')
    ax.set_ylabel('Northing (km)', labelpad=26, fontweight='bold')
    ax.set_zlabel('Depth (m)', labelpad=9, fontweight='bold')
    ax.zaxis.set_major_formatter(FuncFormatter(lambda value, _: f'{value * 1000:.0f}'))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.zaxis.set_major_locator(MaxNLocator(nbins=4))
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.fill = False
        axis.pane.set_edgecolor('#B8B8B8')
    for label in ax.get_xticklabels() + ax.get_yticklabels() + ax.get_zticklabels():
        label.set_fontweight('bold')
    ax.tick_params(pad=2)


In [ ]:
aem = np.load(AEM_RUN / 'stitched_conductivity_section.npz')
keep = np.asarray(aem['chi_squared'], float) <= 2.0
aem_xy = np.asarray(aem['receiver_locations_m'][keep, :2], float)
aem_depth_top = np.asarray(aem['layer_top_depth_m'], float)
aem_sigma = np.asarray(aem['conductivity_s_m'][keep], float)
aem_rho = np.divide(1.0, aem_sigma, out=np.full_like(aem_sigma, np.nan), where=aem_sigma > 0)

geo = np.load(GEO_MODEL, allow_pickle=True)
centres = np.asarray(geo['cell_centres_m'], float)
xc = np.unique(centres[:, 0]); yc = np.unique(centres[:, 1])
nx, ny = len(xc), len(yc)
nz = len(centres) // (nx * ny)
column_z = centres.reshape(nx, ny, nz, 3)[:, :, :, 2]
ground = np.asarray(geo['ground_elevation_m'], float).reshape(nx, ny, nz)
depth_grid = ground - column_z
depthc = depth_grid[0, 0]
if not np.allclose(depth_grid, depthc[None, None, :], atol=1e-6):
    raise ValueError('Model columns do not share a regular depth grid')

# The model file is ordered by column, then increasing depth.
unit = np.asarray(geo['unit'], int).reshape(nx, ny, nz)
cover = np.asarray(geo['cover_depth_m'], float).reshape(nx, ny, nz)[:, :, 0]
doi = np.asarray(geo['investigation_depth_m'], float).reshape(nx, ny, nz)[:, :, 0]
support = np.asarray(geo['support_distance_m'], float).reshape(nx, ny, nz)[:, :, 0]

columns = centres.reshape(nx * ny, nz, 3)[:, 0, :2]
distance, nearest = cKDTree(aem_xy).query(columns)
column_distance = distance.reshape(nx, ny)
layer_index = np.searchsorted(aem_depth_top, depthc, side='right') - 1
layer_index = np.clip(layer_index, 0, aem_depth_top.size - 1)
rho_columns = aem_rho[nearest[:, None], layer_index[None, :]]
rho3 = rho_columns.reshape(nx, ny, nz)
valid3 = ((depthc[None, None, :] <= doi[:, :, None]) &
          (column_distance[:, :, None] <= 300.0) & np.isfinite(rho3))
rho3 = np.where(valid3, rho3, np.nan)

xkm = xc / 1000.0; ykm = yc / 1000.0; zkm = depthc / 1000.0
xedges = cell_edges(xkm); yedges = cell_edges(ykm); zedges = cell_edges(zkm)

# Select two orthogonal sections through the central part of the strongest resistive
# body, avoiding edge-driven choices.
y_candidates = np.arange(max(1, ny // 5), min(ny - 1, ny - ny // 5))
x_candidates = np.arange(max(1, nx // 5), min(nx - 1, nx - nx // 5))
iy = int(y_candidates[np.argmax((unit[:, y_candidates, :] == 2).sum(axis=(0, 2)))])
ix = int(x_candidates[np.argmax((unit[x_candidates, :, :] == 2).sum(axis=(1, 2)))])
xcut, ycut = xkm[ix], ykm[iy]

print(f'AEM soundings retained: {keep.sum():,} / {keep.size:,}')
print(f'Volume grid: {nx} x {ny} x {nz} = {nx*ny*nz:,} cells')
print(f'Longitudinal section: N = {ycut:.3f} km')
print(f'Transverse section: E = {xcut:.3f} km')


## a and c | Reciprocal measurements and error model

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
import case2_cedar_rapids as case2
from ofag.formats.stg import read_stg

SITE_ORDER = ['SVP_S10', 'SVP_CW6', 'Edwd_CW3', 'Ellis_CW1', 'SVPDemo-Day']
SITE_LABEL = {'SVP_S10': 'SVP S10', 'SVP_CW6': 'SVP CW6', 'Edwd_CW3': 'Edwards CW3',
              'Ellis_CW1': 'Ellis CW1', 'SVPDemo-Day': 'SVP Demo-Day'}
KEEP_BLUE, DROP_GREY, MODEL_INK, FIT_BAND = '#00A3E0', '#B3B3B3', '#1F1F1F', '#DAE8F5'
FIT_BLUE = KEEP_BLUE   # shared blue for retained measurements
imports = {item['site']: item for item in STATE['ert_imports']}
case2.ERT = ROOT / 'data/case2_Cedarrapids/ert'   # the case script resolves it from the project root
paths = {site: path for site, array, path in case2._surveys() if array == 'dipole-dipole'}

# Panel a: one quadrupole measured both ways.
fig, axes = plt.subplots(2, 1, figsize=(4.9, 2.9), sharex=True)
for ax, (current, potential, title) in zip(axes, [((0, 1), (4, 5), 'Normal'), ((4, 5), (0, 1), 'Reciprocal')]):
    x = np.arange(6)
    ax.plot([-0.4, 5.4], [0, 0], color='#8C6D46', lw=1.2)
    ax.scatter(x, np.zeros(6), s=60, marker='v', color='#595959', zorder=3)
    for pair, colour, tag in ((current, '#E0762B', 'I'), (potential, KEEP_BLUE, 'V')):
        a, b = pair
        ax.plot([a, a, b, b], [0.05, 0.62, 0.62, 0.05], color=colour, lw=2.0)
        ax.text((a + b) / 2, 0.70, tag, ha='center', va='bottom', fontsize=12, fontweight='bold', color=colour)
    names = dict(zip(current + potential, ['A', 'B', 'M', 'N'])) if title == 'Normal' else dict(zip(potential + current, ['A', 'B', 'M', 'N']))
    for e, name in names.items():
        ax.text(e, -0.22, name, ha='center', va='top', fontsize=11, fontweight='bold')
    ax.text(-0.45, 0.35, title, ha='right', va='center', fontsize=11.5, fontweight='bold')
    ax.set_ylim(-0.6, 1.05); ax.set_xlim(-2.2, 5.6); ax.axis('off')
fig.text(0.97, 0.03, r'$e = |R_\mathrm{n}-R_\mathrm{r}|\,/\,\bar R$', ha='right', fontsize=12)
save_png(fig, 'Fig5a_reciprocal_schematic.png')

# Panel c: every pair, the threshold, and the fitted model.
fig, ax = plt.subplots(figsize=(4.9, 3.7))
r_all = np.logspace(-3, 2, 50)
for site in SITE_ORDER:
    survey = read_stg(paths[site])
    _, resistance, difference = case2._reciprocal_pairs(survey.quadrupoles, survey.resistance_ohm)
    r = np.abs(resistance); refused = difference / r >= case2.RECIPROCAL_ERROR_LIMIT
    ax.scatter(r[~refused], difference[~refused], s=4, color=KEEP_BLUE, alpha=0.25, lw=0, rasterized=True)
    ax.scatter(r[refused], difference[refused], s=9, color=DROP_GREY, marker='x', lw=0.6, alpha=0.8, rasterized=True)
    model = imports[site]['error_model']
    ax.plot(r_all, model['a'] * r_all ** model['b'], color=MODEL_INK, lw=0.8, alpha=0.6)
typical = np.median([imports[s]['error_model']['a'] for s in SITE_ORDER]), np.median([imports[s]['error_model']['b'] for s in SITE_ORDER])
ax.plot(r_all, case2.RECIPROCAL_ERROR_LIMIT * r_all, color='#C55A11', lw=1.4, ls='--')
ax.plot(r_all, np.hypot(typical[0] * r_all ** typical[1], case2.MODELLING_ERROR * r_all), color=MODEL_INK, lw=2.0)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlim(1e-3, 60); ax.set_ylim(1e-7, 30)
ax.set_xlabel(r'Transfer resistance, $|R|$ ($\Omega$)')
ax.set_ylabel(r'Reciprocal error, $|R_\mathrm{n}-R_\mathrm{r}|$ ($\Omega$)')
ax.legend(handles=[Line2D([], [], marker='o', ls='', color=KEEP_BLUE, markersize=5, label='retained'),
                   Line2D([], [], marker='x', ls='', color=DROP_GREY, markersize=6, label='removed'),
                   Line2D([], [], color='#C55A11', lw=1.4, ls='--', label='5% threshold'),
                   Line2D([], [], color=MODEL_INK, lw=0.8, alpha=0.6, label='per-line model'),
                   Line2D([], [], color=MODEL_INK, lw=2.0, label='model + 2%')],
          loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=3, fontsize=9, handlelength=1.6,
          columnspacing=1.0, handletextpad=0.4, prop={'weight': 'bold', 'size': 9})
for side in ('top', 'right'):
    ax.spines[side].set_visible(False)
bold_axes(ax)
save_png(fig, 'Fig5c_error_model.png')
print('error models:', {s: (round(imports[s]['error_model']['a'], 5), round(imports[s]['error_model']['b'], 2)) for s in SITE_ORDER})

## b and d | Filtering and final misfit

In [ ]:
# Panel b: what each step keeps, per line, as readings in the file.
STEPS = [('reciprocal_error_refused', '#F4B183', 'reciprocal error ≥ 5%'),
         ('contact_refused', '#C55A11', 'contact resistance > 2 kΩ'), ('unpaired_refused', '#595959', 'no reciprocal')]
fig, ax = plt.subplots(figsize=(4.9, 3.0))
for y, site in enumerate(SITE_ORDER):
    item = imports[site]
    pairs = {'kept': item['measurements'], 'reciprocal_error_refused': item['reciprocal_error_refused'],
             'contact_refused': item['contact_refused'], 'unpaired_refused': item['unpaired_refused'] / 2}
    total = sum(pairs.values()); left = 0.0
    for key, colour, _ in STEPS:
        width = 100 * pairs[key] / total
        ax.barh(y, width, left=left, color=colour, height=0.62, edgecolor='white', lw=0.8)
        left += width
    ax.text(left + 0.4, y, f"{100 * pairs['kept'] / total:.0f}% retained", va='center', fontsize=10.5, fontweight='bold', color=KEEP_BLUE)
ax.set_yticks(range(len(SITE_ORDER))); ax.set_yticklabels([SITE_LABEL[s] for s in SITE_ORDER])
ax.set_ylim(len(SITE_ORDER) - 0.4, -0.6)
ax.set_xlim(0, 25); ax.set_xticks([0, 5, 10, 15, 20])
ax.set_xlabel('Data removed (%)')
ax.tick_params(axis='y', length=0)
ax.legend(handles=[Patch(color=c, label=l) for _, c, l in STEPS], loc='upper center', bbox_to_anchor=(0.45, -0.3),
          ncol=2, fontsize=9, handlelength=1.0, columnspacing=0.8, prop={'weight': 'bold', 'size': 9})
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
bold_axes(ax)
save_png(fig, 'Fig5b_data_retained.png')

# Panel d: the misfit of each line at its chosen regularization.
fig, ax = plt.subplots(figsize=(4.9, 3.0))
ax.axvspan(0.0, case2.ERT_ACCEPTABLE_CHI_SQUARED, color=FIT_BAND, alpha=0.55, lw=0, zorder=0)
ax.axvline(1.0, color='#595959', lw=1.0, ls='--', zorder=1)
for y, site in enumerate(SITE_ORDER):
    key = f'{site}/dipole-dipole'
    chi2 = float(json.loads((CASE / 'runs' / STATE['ert_runs'][key] / 'result.json').read_text())['summary']['chi_squared'])
    ax.plot([0.0, chi2], [y, y], color='#D9D9D9', lw=1.2, zorder=2)
    ax.scatter(chi2, y, s=54, color=KEEP_BLUE if chi2 <= case2.ERT_ACCEPTABLE_CHI_SQUARED else '#E0762B', edgecolor='white', lw=0.6, zorder=3)
    ax.text(chi2 + 0.08, y, f"λ = {STATE['ert_lambda'][key]:.0f}", va='center', fontsize=10, fontweight='bold', color='#595959')
    print(f'{key:28s} chi2 {chi2:.2f}  lambda {STATE["ert_lambda"][key]:g}')
ax.set_xlim(0, 4.0); ax.set_xticks([0, 1, 2, 3, 4])
ax.set_yticks(range(len(SITE_ORDER))); ax.set_yticklabels([SITE_LABEL[s] for s in SITE_ORDER])
ax.set_ylim(len(SITE_ORDER) - 0.4, -0.6)
ax.tick_params(axis='y', length=0)
ax.set_xlabel(r'Data misfit, $\chi^2$')
for side in ('top', 'right', 'left'):
    ax.spines[side].set_visible(False)
bold_axes(ax)
save_png(fig, 'Fig5d_misfit_lambda.png')

## e | ERT sections

In [ ]:
ERT_NORM = LogNorm(30.0, 2.0e3)
ERT_TICKS = [30, 100, 1000]
ERT_TICKLABELS = ['30', '100', '1000']

ERT_SITES = ['SVP_S10', 'SVP_CW6', 'Ellis_CW1', 'SVPDemo-Day']
DISPLAY_NAMES = {
    'SVP_S10': 'SVP S10', 'SVP_CW6': 'SVP CW6',
    'Ellis_CW1': 'Ellis CW1', 'SVPDemo-Day': 'SVP Demo-Day',
}

sections = []
for site in ERT_SITES:
    run_id = STATE['ert_runs'][f'{site}/dipole-dipole']
    run = CASE / 'runs' / run_id
    mesh = np.load(run / 'mesh_geometry.npz')
    centres2 = np.asarray(mesh['cell_centers_m'], float)
    model = np.asarray(np.load(run / 'recovered_model.npy'), float)
    item = next(row for row in STATE['ert_imports']
                if row['site'] == site and row['array'] == 'dipole-dipole')
    electrodes = np.asarray(np.load(ROOT / item['electrodes_path']), float)
    span = float(np.ptp(electrodes[:, 0]))
    sections.append((site, span, centres2[:, 0], -centres2[:, 1], model))

# Longest to shortest. Ties retain the specified scientific/site order.
sections = sorted(sections, key=lambda item: -item[1])
max_span = max(item[1] for item in sections)
depth_max = 30.0
frame_w_max = 6.35
frame_h = frame_w_max * depth_max / max_span
left, right = 1.05, 1.78
bottom, top, gap = 0.72, 0.32, 0.36
height = bottom + top + len(sections) * frame_h + (len(sections) - 1) * gap
width = left + frame_w_max + right
fig = plt.figure(figsize=(width, height), facecolor='white')

for row, (site, span, distance_m, depth_m, model) in enumerate(sections):
    frame_w = frame_w_max * span / max_span
    x0 = left + (frame_w_max - frame_w) / 2
    y0 = bottom + (len(sections) - 1 - row) * (frame_h + gap)
    ax = fig.add_axes([x0 / width, y0 / height, frame_w / width, frame_h / height])

    # Interpolate log-resistivity from the unstructured inversion mesh onto a regular
    # plotting grid.
    distance_grid = np.linspace(0.0, span, int(round(span)) + 1)
    depth_grid = np.linspace(0.0, depth_max, 121)
    D, Z = np.meshgrid(distance_grid, depth_grid)
    log_model = np.log10(np.clip(model, 1.0, 1.0e4))
    regular = griddata((distance_m, depth_m), log_model, (D, Z), method='linear')
    edge_fill = griddata((distance_m, depth_m), log_model, (D, Z), method='nearest')
    regular = np.where(np.isfinite(regular), regular, edge_fill)
    image = ax.pcolormesh(D, Z, 10.0 ** regular, shading='auto',
                          cmap=RES_CMAP, norm=ERT_NORM, rasterized=True)
    ax.plot([0, span], [0, 0], color='#151515', lw=1.1, zorder=5)
    ax.set_xlim(0, span); ax.set_ylim(depth_max, 0)
    ax.set_aspect('equal', adjustable='box')
    ax.set_ylabel('Depth (m)')
    if row == len(sections) - 1:
        ax.set_xlabel('Distance (m)')
    else:
        ax.set_xticklabels([])
    ax.text(0.0, 1.035, DISPLAY_NAMES[site], transform=ax.transAxes,
            ha='left', va='bottom', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
    bold_axes(ax)

cax = fig.add_axes([(left + frame_w_max + 0.20) / width,
                    bottom / height, 0.27 / width,
                    (len(sections) * frame_h + (len(sections) - 1) * gap) / height])
cb = fig.colorbar(ScalarMappable(norm=ERT_NORM, cmap=RES_CMAP), cax=cax)
cb.set_ticks(ERT_TICKS)
cb.set_ticklabels(ERT_TICKLABELS)
cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
cb.ax.minorticks_off()
cb.ax.tick_params(labelsize=11, width=1.2, length=5)
for label in cb.ax.get_yticklabels():
    label.set_fontweight('bold')
save_png(fig, 'Fig5e_ERT_sections.png', tight=False)
print([(DISPLAY_NAMES[s], span) for s, span, *_ in sections])


## f | ERT and AEM depth comparison

In [ ]:
import sys
sys.path.insert(0, str(ROOT / 'scripts'))
import case2_cedar_rapids
from case2_cedar_rapids import _site_positions
if not case2_cedar_rapids.AEM.is_absolute():
    case2_cedar_rapids.AEM = ROOT / case2_cedar_rapids.AEM
COVERAGE_BAR_DECADES = 2.0
AEM_REACH_FOR_PROFILES_M = 300.0
ERT_COLOR, AEM_COLOR = '#C8553D', '#2A5C8A'
frame_w, frame_h = 7.2, 2.55
left, right, bottom, top = 1.05, 0.30, 1.10, 0.62
width, height = left + frame_w + right, bottom + frame_h + top
# ---------------------------------------------------------------- panel e
aem_fit_xy = aem_xy                           # the fitting soundings, as in panel a
profile_depths = np.arange(2.5, 50.1, 2.5)
band = 1.25
ert_values = {d: [] for d in profile_depths}
aem_values = {d: [] for d in profile_depths}
drop_values = {d: [] for d in profile_depths}
aem_tree = cKDTree(aem_fit_xy)
site_counts = {}
for site in ERT_SITES:
    run = CASE / 'runs' / STATE['ert_runs'][f'{site}/dipole-dipole']
    item = next(r for r in STATE['ert_imports'] if r['site'] == site and r['array'] == 'dipole-dipole')
    along = np.asarray(np.load(ROOT / item['electrodes_path']), float)[:, 0]
    near = sorted({j for hits in aem_tree.query_ball_point(_site_positions(site, along),
                                                             AEM_REACH_FOR_PROFILES_M) for j in hits})
    mesh = np.load(run / 'mesh_geometry.npz')['cell_centers_m']
    model = np.asarray(np.load(run / 'recovered_model.npy'), float)
    coverage = np.asarray(np.load(run / 'model_coverage.npy'), float)
    on_line = (mesh[:, 0] >= along.min()) & (mesh[:, 0] <= along.max())
    site_counts[site] = (int(on_line.sum()), len(near))
    for d in profile_depths:
        take = on_line & (np.abs(-mesh[:, 1] - d) <= band)
        ert_values[d].extend(model[take])
        drop_values[d].extend(coverage.max() - coverage[take])
        layer = np.searchsorted(aem_depth_top, d, side='right') - 1
        aem_values[d].extend(aem_rho[near, layer])

def quartiles(values):
    return np.array([np.percentile(values[d], [25, 50, 75]) for d in profile_depths])

ert_q, aem_q = quartiles(ert_values), quartiles(aem_values)
median_drop = np.array([np.median(drop_values[d]) for d in profile_depths])
ratio = ert_q[:, 1] / aem_q[:, 1]
beyond = profile_depths[np.flatnonzero(median_drop > COVERAGE_BAR_DECADES)[0]] - band

# The ERT is drawn down to the last band whose coverage holds, and not below: under that
# it is its smoothness term speaking, not its data.
ert_valid = np.arange(profile_depths.size) < np.flatnonzero(median_drop > COVERAGE_BAR_DECADES)[0]

# The same canvas and plot box as panel d, so the two stack with their frames aligned.
fig = plt.figure(figsize=(width, height), facecolor='white')
ax = fig.add_axes([left / width, bottom / height, frame_w / width, frame_h / height])
ax.axvspan(beyond, 50.0, color='#E4E6E9', zorder=0, lw=0)
for q, keep_rows, color, name in ((aem_q, np.ones(profile_depths.size, bool), AEM_COLOR, 'AEM'),
                                  (ert_q, ert_valid, ERT_COLOR, 'ERT')):
    keep_rows = keep_rows & (profile_depths >= 10.0)   # only what the axis shows, so nothing is drawn past it
    depth_rows = profile_depths[keep_rows]
    ax.fill_between(depth_rows, q[keep_rows, 0], q[keep_rows, 2], color=color, alpha=0.18, lw=0, zorder=2)
    ax.plot(depth_rows, q[keep_rows, 1], color=color, lw=2.2, zorder=3, label=name)
ax.set_yscale('log'); ax.set_ylim(30, 2000)
ax.set_yticks(ERT_TICKS); ax.set_yticklabels(ERT_TICKLABELS)
ax.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax.set_xlim(10.0, 50.0)                        # the depths both methods are read over
ax.set_xticks([10, 20, 30, 40, 50])
ax.set_xlabel('Depth (m)'); ax.set_ylabel(r'Resistivity ($\Omega$ m)')
ax.legend(loc='upper left', fontsize=11, handlelength=1.8)
bold_axes(ax)
save_png(fig, 'Fig5f_ERT_AEM_depth_profiles.png', tight=False)

print('ERT cells on the line / AEM soundings within 300 m:', site_counts)
print('depth  ERT median  AEM median  ratio  median ERT coverage drop')
for d, e, a, r, c in zip(profile_depths, ert_q[:, 1], aem_q[:, 1], ratio, median_drop):
    print(f'{d:5.1f}  {e:10.1f}  {a:10.1f}  {r:5.2f}  {c:5.2f}')
print(f'ERT coverage passes the {COVERAGE_BAR_DECADES:.0f}-decade bar below {beyond:.1f} m')

## Export check

Every listed panel must exist and have nonzero dimensions.

In [ ]:
from PIL import Image
expected = ['Fig5a_reciprocal_schematic.png', 'Fig5b_data_retained.png', 'Fig5c_error_model.png', 'Fig5d_misfit_lambda.png', 'Fig5e_ERT_sections.png', 'Fig5f_ERT_AEM_depth_profiles.png']
for name in expected:
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as panel:
        assert panel.width > 0 and panel.height > 0, path
print(f'Figure panels checked: {len(expected)} in {OUT}')
